Real Life Testing

In [3]:
import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import json
import time
from collections import deque
import torch_directml
from scipy.interpolate import interp1d
import warnings

warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. 9-LAYER CORE ARCHITECTURE (EMBEDDING EXTRACTOR)
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [(0,1), (1,2), (2,3), (3,7), (0,4), (4,5), (5,6), (6,8), (9,10), (11,12), (11,23), (12,24), (23,24), (11,13), (13,15), (12,14), (14,16), (15,17), (15,19), (15,21), (16,18), (16,20), (16,22)]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4), (0,5), (5,6), (6,7), (7,8), (5,9), (9,10), (10,11), (11,12), (9,13), (13,14), (14,15), (15,16), (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges: A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        self.attention = ChannelAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
        
    def forward(self, x): 
        sgcn_out = self.sgcn(x)
        tgcn_out = self.tgcn(sgcn_out)
        attended_out = self.attention(tgcn_out)
        return F.relu(attended_out + self.res(x)) 

class BanglaSignSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        # 🟢 RETURNING RAW 256-D EMBEDDING FOR DATABASE MATCHING
        return x.view(x.size(0), -1)

# ==========================================
# 2. MEDIAPIPE EXTRACTOR (VALIDATED MATH)
# ==========================================
def extract_keypoints(results):
    frame_data = np.zeros((68, 3))

    if results.pose_landmarks:
        for i in range(23):
            lm = results.pose_landmarks.landmark[i]
            frame_data[i] = [lm.x, lm.y, lm.z]
        lm_l_hip = results.pose_landmarks.landmark[23]
        frame_data[23] = [lm_l_hip.x, lm_l_hip.y, lm_l_hip.z]
        lm_r_hip = results.pose_landmarks.landmark[24]
        frame_data[24] = [lm_r_hip.x, lm_r_hip.y, lm_r_hip.z]

    if results.face_landmarks:
        lm_chin = results.face_landmarks.landmark[152]
        frame_data[25] = [lm_chin.x, lm_chin.y, lm_chin.z]

    if results.left_hand_landmarks:
        for i, lm in enumerate(results.left_hand_landmarks.landmark):
            frame_data[26 + i] = [lm.x, lm.y, lm.z]

    if results.right_hand_landmarks:
        for i, lm in enumerate(results.right_hand_landmarks.landmark):
            frame_data[47 + i] = [lm.x, lm.y, lm.z]

    return frame_data

def format_for_ui(raw_prediction):
    clean_name = raw_prediction.replace('_Righthand', '').replace('_Lefthand', '')
    if clean_name == '0' or clean_name == 'O': return '0 / O'
    if clean_name == '2' or clean_name == 'V': return '2 / V'
    clean_name = clean_name.replace('_', ' ')
    return clean_name

# ==========================================
# 3. LIVE 9-LAYER DATABASE ENGINE
# ==========================================
def run_live_database_inference():
    # 🟢 PATHS
    # Ensure these point to the weights and database generated by the 9-LAYER MODEL
    EXPERIMENT_FOLDER = r"C:\Users\User\Documents\Personal Akams\Thesis\1.Making the model\Baselinemodel Code Experiments\7.9_layer_Attention_Models"
    MODEL_WEIGHTS = os.path.join(EXPERIMENT_FOLDER, "9_layer_attention_relu_9frame.pth")
    
    # Update these paths to where you saved your 9-layer .npy database
    DATABASE_FOLDER = r"C:\Users\User\Documents\Personal Akams\Thesis\1.Making the model\Baselinemodel Code Experiments\database Saved"
    DB_PATH = os.path.join(DATABASE_FOLDER, "class_prototypes.npy") 
    LABELS_PATH = os.path.join(DATABASE_FOLDER, "prototype_labels.json")

    CONFIDENCE_THRESHOLD = 0.50
    SOFTMAX_TEMPERATURE = 2.0  
    MOTION_THRESHOLD = 0.015
    STILL_FRAMES_TO_END = 20
    MIN_GESTURE_FRAMES = 15
    MAX_GESTURE_FRAMES = 150
    MAX_CONSECUTIVE_DROPS = 10   
    RESULT_DISPLAY_FRAMES = 90
    LEFT_SHOULDER_IDX = 11
    RIGHT_SHOULDER_IDX = 12

    target_idx = 0
    print("\n🔍 Scanning available DirectML GPUs...")
    for i in range(torch_directml.device_count()):
        gpu_name = torch_directml.device_name(i)
        if "7900" in gpu_name or "GRE" in gpu_name or "RX" in gpu_name: target_idx = i

    dml = torch_directml.device(target_idx)
    print(f"🚀 9-LAYER DATABASE ENGINE BOOTED ON: {torch_directml.device_name(target_idx)}")

    # 🟢 LOAD DATABASE
    if not os.path.exists(DB_PATH) or not os.path.exists(LABELS_PATH):
        print(f"❌ ERROR: Database files not found in {DATABASE_FOLDER}. Please build the 9-layer database first.")
        return

    prototypes_matrix = np.load(DB_PATH) 
    with open(LABELS_PATH, "r") as f: labels_list = json.load(f) 
    
    unique_classes = sorted(list(set(labels_list)))
    num_classes = len(unique_classes)
    prototypes_tensor = torch.tensor(prototypes_matrix, dtype=torch.float32).to(dml)

    # 🟢 LOAD MODEL
    model = BanglaSignSTGCN(num_classes, Graph().A).to(dml)
    pretrained_dict = torch.load(MODEL_WEIGHTS, map_location=dml, weights_only=False)
    model_dict = model.state_dict()
    pretrained_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict}
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)
    model.eval()
    print("✅ Model & Database Ready.")

    mp_holistic = mp.solutions.holistic
    holistic = mp_holistic.Holistic(static_image_mode=False, model_complexity=2, min_detection_confidence=0.5, min_tracking_confidence=0.5)
    mp_drawing = mp.solutions.drawing_utils
    landmark_style = mp_drawing.DrawingSpec(color=(0, 255, 255), thickness=1, circle_radius=1)
    connection_style = mp_drawing.DrawingSpec(color=(255, 0, 255), thickness=2)

    def predict_database_sequence(keypoints_list):
        sequence = np.array(keypoints_list)  
        T = sequence.shape[0]

        if T == 90:
            interpolated = sequence
        elif T == 1:
            interpolated = np.repeat(sequence, 90, axis=0)
        else:
            x_original = np.linspace(0, 1, T)
            x_target = np.linspace(0, 1, 90)
            interpolator = interp1d(x_original, sequence, axis=0, kind='linear')
            interpolated = interpolator(x_target)

        left_shoulder = interpolated[0, LEFT_SHOULDER_IDX]
        right_shoulder = interpolated[0, RIGHT_SHOULDER_IDX]
        chest_midpoint = (left_shoulder + right_shoulder) / 2.0
        sequence_centered = interpolated - chest_midpoint

        ls_2d = sequence_centered[0, LEFT_SHOULDER_IDX, :2]
        rs_2d = sequence_centered[0, RIGHT_SHOULDER_IDX, :2]
        shoulder_width = np.linalg.norm(ls_2d - rs_2d)
        sequence_aligned = sequence_centered / shoulder_width if shoulder_width > 0.01 else sequence_centered

        sequence_final = sequence_aligned - np.mean(sequence_aligned, axis=1, keepdims=True)
        tensor_seq = torch.tensor(sequence_final, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(dml)

        with torch.no_grad():
            embedding = model(tensor_seq)
            # Distance to all database prototypes
            distances = torch.cdist(embedding, prototypes_tensor, p=2).squeeze(0).cpu().numpy()

        # This logic handles BOTH single-matrix (1 index per class) and multi-cluster databases
        class_distances = np.zeros(len(unique_classes))
        for c_idx, class_name in enumerate(unique_classes):
            target_indices = [i for i, lbl in enumerate(labels_list) if lbl == class_name]
            class_distances[c_idx] = np.min(distances[target_indices])

        # Convert Euclidean distance to Probability via Softmax
        neg_sq_dist = -(class_distances ** 2) / SOFTMAX_TEMPERATURE
        neg_sq_dist = neg_sq_dist - neg_sq_dist.max()  
        exp_scores = np.exp(neg_sq_dist)
        class_probs = exp_scores / exp_scores.sum()

        sorted_indices = np.argsort(class_probs)[::-1][:5]
        return [unique_classes[i] for i in sorted_indices], [class_probs[i] for i in sorted_indices]

    # State
    idle_buffer = deque(maxlen=5)
    gesture_buffer = []
    motion_state = "IDLE"
    prev_hand_points = None
    still_frame_counter = 0
    consecutive_drop_streak = 0
    result_display_counter = 0
    motion_score = 0.0

    current_prediction = "Waiting for sign..."
    confidence_score = 0.0
    top_5_classes = []
    top_5_probs = []
    color = (200, 200, 200)

    calibration_frames = 0
    is_calibrated = False

    print("⏳ Warming up MediaPipe engine...")
    blank_image = np.zeros((480, 640, 3), dtype=np.uint8)
    _ = holistic.process(blank_image)

    # Booting Camera safely
    cap = None
    for idx in [0, 1, 2]:
        test_cap = cv2.VideoCapture(idx, cv2.CAP_DSHOW)
        time.sleep(0.2) 
        if test_cap.isOpened():
            ret, _ = test_cap.read()
            if ret: cap = test_cap; break
        test_cap.release()

    if cap is None: 
        for idx in [0, 1, 2]:
            test_cap = cv2.VideoCapture(idx)
            time.sleep(0.2) 
            if test_cap.isOpened():
                ret, _ = test_cap.read()
                if ret: cap = test_cap; break
            test_cap.release()

    if cap is None or not cap.isOpened(): return

    cap.set(cv2.CAP_PROP_FPS, 30)
    print("🎥 Live UI Ready.")

    PANEL_WIDTH = 450
    fps_frame_times = []
    fps_display = 0.0
    FPS_WINDOW = 30

    while cap.isOpened():
        loop_start_time = time.time()
        ret, frame = cap.read()
        if not ret: break

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(image_rgb)

        fps_frame_times.append(loop_start_time)
        if len(fps_frame_times) > FPS_WINDOW: fps_frame_times.pop(0)
        if len(fps_frame_times) >= 2:
            elapsed = fps_frame_times[-1] - fps_frame_times[0]
            if elapsed > 0: fps_display = (len(fps_frame_times) - 1) / elapsed

        if results.pose_landmarks: mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, landmark_style, connection_style)
        if results.left_hand_landmarks: mp_drawing.draw_landmarks(frame, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, landmark_style, connection_style)
        if results.right_hand_landmarks: mp_drawing.draw_landmarks(frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, landmark_style, connection_style)

        h, w, _ = frame.shape
        dashboard = np.zeros((h, PANEL_WIDTH, 3), dtype=np.uint8)

        if not is_calibrated:
            cx, cy = int(frame.shape[1] / 2), int(frame.shape[0] / 2)
            cv2.rectangle(frame, (cx - 180, cy - 150), (cx + 180, cy + 180), (0, 0, 255), 2)

            if results.pose_landmarks:
                nose = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.NOSE]
                if 0.35 < nose.x < 0.65 and 0.15 < nose.y < 0.45:
                    calibration_frames += 1
                    cv2.putText(frame, f"Steady... {calibration_frames}/30", (cx - 70, cy - 160), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                    cv2.rectangle(frame, (cx - 180, cy - 150), (cx + 180, cy + 180), (0, 255, 255), 2)
                else:
                    calibration_frames = 0

            if calibration_frames >= 30: is_calibrated = True
            cv2.putText(dashboard, "STATUS: CALIBRATING...", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        else:
            if result_display_counter == 0:
                is_tracked = results.pose_landmarks is not None and (
                    results.left_hand_landmarks is not None or results.right_hand_landmarks is not None
                )

                if is_tracked:
                    keypoints = extract_keypoints(results)
                    consecutive_drop_streak = 0

                    hand_points = keypoints[26:68]
                    if prev_hand_points is not None:
                        motion_score = float(np.mean(np.linalg.norm(hand_points - prev_hand_points, axis=1)))
                    else:
                        motion_score = 0.0
                    prev_hand_points = hand_points.copy()

                    if motion_state == "IDLE":
                        idle_buffer.append(keypoints)
                        if motion_score > MOTION_THRESHOLD:
                            motion_state = "🔴 RECORDING..."
                            gesture_buffer = list(idle_buffer)
                            still_frame_counter = 0
                    else:  
                        if len(gesture_buffer) < MAX_GESTURE_FRAMES: gesture_buffer.append(keypoints)
                        if motion_score <= MOTION_THRESHOLD: still_frame_counter += 1
                        else: still_frame_counter = 0
                else:
                    consecutive_drop_streak += 1

                if motion_state != "IDLE":
                    gesture_finished = still_frame_counter >= STILL_FRAMES_TO_END
                    gesture_capped = len(gesture_buffer) >= MAX_GESTURE_FRAMES
                    gesture_lost = consecutive_drop_streak > MAX_CONSECUTIVE_DROPS

                    if gesture_lost:
                        current_prediction = "Lost tracking - try again"
                        color = (0, 165, 255)
                        result_display_counter = RESULT_DISPLAY_FRAMES
                        motion_state = "IDLE"
                        gesture_buffer = []
                        still_frame_counter = 0
                        consecutive_drop_streak = 0

                    elif gesture_finished or gesture_capped:
                        if len(gesture_buffer) >= MIN_GESTURE_FRAMES:
                            motion_state = "⚙️ PROCESSING..."
                            
                            # 🟢 RUN 9-LAYER DATABASE LOGIC
                            top_5_classes, top_5_probs = predict_database_sequence(gesture_buffer)
                            best_class = top_5_classes[0]
                            confidence_score = top_5_probs[0]

                            print(f"\n[DEBUG] Evaluated against 9-Layer Database (Softmax Temp {SOFTMAX_TEMPERATURE})")
                            for c, p in zip(top_5_classes, top_5_probs):
                                print(f"   {c:30s} {p*100:5.1f}%")

                            if confidence_score >= CONFIDENCE_THRESHOLD:
                                current_prediction = format_for_ui(best_class)
                                color = (0, 255, 0)
                            else:
                                current_prediction = "Low Confidence"
                                color = (0, 165, 255)

                            result_display_counter = RESULT_DISPLAY_FRAMES

                        motion_state = "IDLE"
                        gesture_buffer = []
                        still_frame_counter = 0
                        consecutive_drop_streak = 0

            if result_display_counter > 0:
                result_display_counter -= 1
                if result_display_counter == 0:
                    current_prediction = "Waiting to detect motion..."
                    top_5_classes = []
                    color = (200, 200, 200)

            status_color = (0, 0, 255) if motion_state == "🔴 RECORDING..." else (0, 255, 255) if motion_state == "⚙️ PROCESSING..." else (150, 150, 150)
            cv2.putText(dashboard, f"STATUS: {motion_state}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
            cv2.line(dashboard, (20, 60), (PANEL_WIDTH - 20, 60), (100, 100, 100), 1)

            cv2.putText(dashboard, "9-LAYER DATABASE PREDICTION:", (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
            cv2.putText(dashboard, current_prediction, (20, 140), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
            cv2.putText(dashboard, f"Database Conf: {confidence_score*100:.1f}%", (20, 175), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1)

            if top_5_classes and confidence_score < CONFIDENCE_THRESHOLD:
                cv2.line(dashboard, (20, 210), (PANEL_WIDTH - 20, 210), (100, 100, 100), 1)
                cv2.putText(dashboard, "TOP 5 MATCHES:", (20, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 1)
                y_offset = 280
                for i in range(len(top_5_classes)):
                    text = f"{i+1}. {format_for_ui(top_5_classes[i])[:20]}... {top_5_probs[i]*100:.1f}%"
                    cv2.putText(dashboard, text, (20, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
                    y_offset += 35

        cv2.putText(dashboard, f"FPS: {fps_display:.1f}", (20, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1)

        combined_view = np.hstack((frame, dashboard))
        cv2.imshow('Live Sign Translation - 9L Database', combined_view)

        key = cv2.waitKey(10) & 0xFF
        if key == ord('q'): break
        if key == ord('r'):
            is_calibrated = False
            calibration_frames = 0
            gesture_buffer = []
            motion_state = "IDLE"
            prev_hand_points = None
            consecutive_drop_streak = 0
            idle_buffer.clear()
            current_prediction = "Waiting for sign..."
            confidence_score = 0.0
            top_5_classes = []
            color = (200, 200, 200)

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_live_database_inference()


🔍 Scanning available DirectML GPUs...
🚀 9-LAYER DATABASE ENGINE BOOTED ON: AMD Radeon RX 7900 GRE 
✅ Model & Database Ready.
⏳ Warming up MediaPipe engine...
🎥 Live UI Ready.


c:\Users\User\Envs\cvpr_master_env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



[DEBUG] Evaluated against 9-Layer Database (Softmax Temp 2.0)
   Balti_Water Bucket_Righthand    95.1%
   Chips_Chips_Righthand            4.9%
   TV_Television_Righthand          0.0%
   College_College_Righthand        0.0%
   Dal_Lentils_Righthand            0.0%

[DEBUG] Evaluated against 9-Layer Database (Softmax Temp 2.0)
   5_Righthand                    100.0%
   Hello_Hello_Righthand            0.0%
   Shundor_Nice_Righthand           0.0%
   Bahire_Outside_Righthand         0.0%
   Office_Office_Righthand          0.0%

[DEBUG] Evaluated against 9-Layer Database (Softmax Temp 2.0)
   00_Righthand                    82.7%
   000_Righthand                   12.9%
   Betha_Hurts_Righthand            2.7%
   V_Righthand                      0.7%
   Office_Office_Righthand          0.6%

[DEBUG] Evaluated against 9-Layer Database (Softmax Temp 2.0)
   Dhonnobad_Thanks_Righthand      61.2%
   Dal_Lentils_Righthand           21.5%
   Betha_Hurts_Righthand           16.6%
   4_Right